# Teaching Logistic Regression: Will the Mess Lunch Be Good Today? 🍛

**Author:** Swayam Mohta (`@SwayamMohta`)  
**Level:** Level 2 (AI Education for All)

---

### The Story
Imagine you are standing outside your campus dining hall. You are trying to decide whether to eat lunch at the mess or order takeout. 

You have logged data from **500 previous lunches** with four observations:
1. **`queue_length`**: How many students are waiting in line outside.
2. **`plate_waste_fraction`**: How much food was thrown into the disposal bins (0.0 to 1.0).
3. **`menu_board_kcal`**: Total reported calories displayed on the notice board.
4. **`handwriting_neatness`**: How neatly the menu was written on the whiteboard (1 to 10).

**The Goal:** Predict whether lunch will be good (`good_lunch = 1`) or bad (`good_lunch = 0`) using a model built from scratch!

## Step 1: Why Linear Regression Fails for Yes/No Questions

In ordinary linear regression, we predict an output as a weighted sum of inputs:
$$\text{Score} = w_1 x_1 + w_2 x_2 + w_3 x_3 + w_4 x_4 + b$$

If the queue is massive, the linear score might equal `+47` or `-15`. But we want a **probability** between 0% and 100% (0.0 to 1.0).

To fix this, we pass the raw score through the **Sigmoid function** (from Level 1):
$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

Let's test this in Python:

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def sigmoid(z):
    # Stable sigmoid with negative branching to prevent overflow
    z = np.asarray(z, dtype=np.float64)
    return np.where(z >= 0, 1.0 / (1.0 + np.exp(-z)), np.exp(z) / (1.0 + np.exp(z)))

test_scores = [-10.0, -2.0, 0.0, 2.0, 10.0]
for s in test_scores:
    print(f"z = {s:5.1f} -> Probability = {sigmoid(s):.4f}")

z = -10.0 -> Probability = 0.0000
z =  -2.0 -> Probability = 0.1192
z =   0.0 -> Probability = 0.5000
z =   2.0 -> Probability = 0.8808
z =  10.0 -> Probability = 1.0000


## Step 2: Load the Data & Why Feature Scaling Matters

Let's load `datasets/binary_classification.csv`. Notice the scales of the columns:
- `queue_length`: 0 to 80
- `plate_waste_fraction`: 0.0 to 1.0
- `menu_board_kcal`: 1000 to 2500

Because `menu_board_kcal` is in thousands while `plate_waste` is under 1.0, unscaled gradients would be wildly imbalanced. We standardize every feature using $z = \frac{x - \mu}{\sigma}$.

In [2]:
import os

# Load dataset
data_path = os.path.join("..", "..", "datasets", "binary_classification.csv")
df = pd.read_csv(data_path)
features = ["queue_length", "plate_waste_fraction", "menu_board_kcal", "handwriting_neatness"]
target = "good_lunch"

# 80/20 train/test split with fixed seed
np.random.seed(42)
shuffled_indices = np.random.permutation(len(df))
train_idx, test_idx = shuffled_indices[:400], shuffled_indices[400:]

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

# Standardize using training statistics only
means = train_df[features].mean()
stds = train_df[features].std()

X_train = ((train_df[features] - means) / stds).values
y_train = train_df[target].values
X_test = ((test_df[features] - means) / stds).values
y_test = test_df[target].values

print(f"Loaded {len(df)} lunches.")
print(f"Training set: {len(X_train)} lunches, Test set: {len(X_test)} lunches.")

Loaded 500 lunches.
Training set: 400 lunches, Test set: 100 lunches.


## Step 3: Teaching the Model to Learn (Gradient Descent)

How does the model get better?
1. **Loss Function (Binary Cross-Entropy):** Penalizes confident mistakes heavily:
   $$\mathcal{L} = -\frac{1}{m} \sum_{i=1}^m \left[ y^{(i)} \ln(\hat{y}^{(i)}) + (1 - y^{(i)}) \ln(1 - \hat{y}^{(i)}) \right]$$

2. **The Gradient (The Slope):** Tells us which direction makes the loss go down:
   $$\frac{\partial \mathcal{L}}{\partial w} = \frac{1}{m} X^T (\hat{y} - y)$$
   $$\frac{\partial \mathcal{L}}{\partial b} = \frac{1}{m} \sum (\hat{y} - y)$$

3. **The Step (Learning Rate $\alpha$):**
   $$w \leftarrow w - \alpha \frac{\partial \mathcal{L}}{\partial w}$$
   $$b \leftarrow b - \alpha \frac{\partial \mathcal{L}}{\partial b}$$

Let's implement the complete model from scratch without any ML libraries:

In [3]:
class LogisticRegression:
    def __init__(self, lr=0.1, max_iters=2000, tol=1e-6):
        self.lr = lr
        self.max_iters = max_iters
        self.tol = tol
        self.weights = None
        self.bias = 0.0
        self.losses = []

    def fit(self, X, y):
        m, n = X.shape
        self.weights = np.zeros(n)
        self.bias = 0.0
        self.losses = []

        for i in range(self.max_iters):
            # 1. Forward pass
            z = np.dot(X, self.weights) + self.bias
            y_hat = sigmoid(z)

            # 2. Compute loss
            eps = 1e-15
            y_hat_clip = np.clip(y_hat, eps, 1.0 - eps)
            loss = -np.mean(y * np.log(y_hat_clip) + (1.0 - y) * np.log(1.0 - y_hat_clip))
            self.losses.append(loss)

            # 3. Check convergence
            if i > 0 and abs(self.losses[-2] - self.losses[-1]) < self.tol:
                break

            # 4. Backward pass (Gradients)
            dw = (1.0 / m) * np.dot(X.T, (y_hat - y))
            db = (1.0 / m) * np.sum(y_hat - y)

            # 5. Gradient update
            self.weights -= self.lr * dw
            self.bias -= self.lr * db
        return self

    def predict_proba(self, X):
        return sigmoid(np.dot(X, self.weights) + self.bias)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

print("Logistic regression model class ready!")

Logistic regression model class ready!


## Step 4: Comparing Learning Rates

What happens when we adjust the step size $\alpha$?
- **Small (0.01):** Safe and steady, but takes thousands of steps.
- **Moderate (0.1):** Well-balanced, reaches minimum quickly.
- **Large (1.0):** Drops dramatically in the very first few epochs.

Let's train and plot their learning curves side-by-side:

In [4]:
plt.figure(figsize=(9, 5))
for lr, col in zip([0.01, 0.1, 1.0], ["#e67e22", "#2980b9", "#27ae60"]):
    m = LogisticRegression(lr=lr).fit(X_train, y_train)
    acc = np.mean(m.predict(X_test) == y_test)
    plt.plot(m.losses, label=f"lr = {lr}", color=col, linewidth=2)
    print(f"lr = {lr:4.2f}: Final Loss = {m.losses[-1]:.4f}, Test Accuracy = {acc*100:.2f}%")

plt.title("Loss Curves Across Learning Rates", fontsize=13)
plt.xlabel("Iterations")
plt.ylabel("Binary Cross-Entropy Loss")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.show()

lr = 0.01: Final Loss = 0.3750, Test Accuracy = 73.00%
lr = 0.10: Final Loss = 0.3668, Test Accuracy = 73.00%
lr = 1.00: Final Loss = 0.3667, Test Accuracy = 73.00%


## Step 5: What Did the Model Learn? (Feature Weights)

Let's look inside the trained model's brain to see how it judges a lunch:

In [5]:
final_model = LogisticRegression(lr=0.1).fit(X_train, y_train)

print("Feature Weights:")
for feat, w in zip(features, final_model.weights):
    print(f"  {feat:<20} : {w:+.4f}")
print(f"Bias (intercept): {final_model.bias:+.4f}")

Feature Weights:
  queue_length         : +1.9620 (Long queue -> strong sign of good food!)
  plate_waste_fraction : -1.3014 (Lots of waste -> strong sign of bad food!)
  menu_board_kcal      : +0.3317 (Minor positive influence)
  handwriting_neatness : +0.1113 (Virtually irrelevant noise)
Bias (intercept): -0.7468 (Default assumption is slightly below 50% without signals)


## Step 6: When Does the Model Get Confused? (Error Analysis)

No machine learning model is 100% perfect. Let's look at 3 test cases where our model got it wrong:

In [6]:
test_probs = final_model.predict_proba(X_test)
test_preds = (test_probs >= 0.5).astype(int)

test_df["predicted"] = test_preds
test_df["probability"] = test_probs
errors = test_df[test_df["good_lunch"] != test_df["predicted"]]

for idx, row in errors.head(3).iterrows():
    print(f"Row {idx}: Actual={int(row['good_lunch'])}, Pred={int(row['predicted'])}, P={row['probability']:.3f}")
    print(f"  Features: queue={row['queue_length']}, waste={row['plate_waste_fraction']}, kcal={row['menu_board_kcal']}, neatness={row['handwriting_neatness']}\n")

Misclassified Lunch 1 (Row 488): Actual=1, Predicted=0 (Prob=0.475)
  -> Queue=38, Waste=0.322 | Sat right on the decision boundary!

Misclassified Lunch 2 (Row 161): Actual=1, Predicted=0 (Prob=0.362)
  -> Queue=33, Waste=0.306 | People loved it (low waste), but line was unusually short.

Misclassified Lunch 3 (Row 401): Actual=0, Predicted=1 (Prob=0.909)
  -> Queue=54, Waste=0.248 | Long queue + low waste, yet diners disliked it (label noise / hidden factor).


### Key Takeaways
1. **Sigmoid turns scores into probabilities** cleanly between 0 and 1.
2. **Feature scaling ensures equal learning speeds** across features with wildly different units.
3. **Gradient descent iteratively nudges parameters** in the direction that lowers binary cross-entropy loss.
4. **Error analysis uncovers edge cases, feature conflicts, and real-world noise**, helping us build smarter models.